# Lab: Gọi API Xổ Số Miền Bắc (XSMB)

**Nguồn API:** https://xosoapi.online/ket-qua-xo-so/mien-bac

Notebook này minh hoạ quy trình **thực tế** khi làm việc với một API bên thứ ba:

1. Đọc tài liệu API để hiểu endpoint, cách xác thực, tham số.
2. Thử nghiệm nhanh bằng `curl`.
3. Viết code Python (`requests`) để gọi API, xử lý lỗi và phân trang.
4. Lưu dữ liệu ra file trung gian, sau đó xử lý bằng `pandas`.

## 0. Đọc tài liệu API — cần chú ý những gì?

Trước khi viết bất kỳ dòng code nào, hãy đọc tài liệu (docs) của API và xác định:

| Mục | Ý nghĩa | Ví dụ với API này |
|---|---|---|
| **Base URL / Endpoint** | Địa chỉ gốc để gọi API | `https://xosoapi.online/api/v1/vietnam/draws` |
| **Method** | GET / POST / PUT... | `GET` (chỉ lấy dữ liệu) |
| **Xác thực (Auth)** | API yêu cầu key/token qua đâu? | Header `X-API-Key: <key>` |
| **Query params** | Tham số lọc/phân trang | `page`, `limit` |
| **Response schema** | Cấu trúc JSON trả về | `{ success, data: [...], meta: {...} }` |
| **Phân trang (pagination)** | Làm sao biết còn dữ liệu để lấy tiếp? | `meta.total`, `meta.page`, `meta.last_page` |
| **Rate limit** | Giới hạn số lượt gọi | Gọi dồn dập sẽ bị lỗi `429 Too Many Requests` |

Việc xác định trước các mục này giúp bạn viết code đúng ngay từ đầu, thay vì đoán mò.

In [ ]:
# Cài thư viện nếu môi trường chưa có (bỏ comment dòng dưới nếu cần)
# %pip install requests pandas

API_KEY = "YOUR_API_KEY_HERE"
API_URL = "https://xosoapi.online/api/v1/vietnam/draws"

## 1. Test API bằng `curl`

`curl` là công cụ dòng lệnh để gọi HTTP request nhanh, **không cần viết code**. Đây thường là bước đầu tiên khi tiếp cận một API mới: xác nhận key đúng, xem thử response thật trông như thế nào, trước khi đầu tư viết code Python xử lý nó.

In [ ]:
!curl -H "X-API-Key: {API_KEY}" "https://xosoapi.online/api/v1/vietnam/draws"

## 2. Lưu response ra file JSON

Ghép `curl` với `python -m json.tool` để format lại JSON cho dễ đọc, rồi `>` để lưu ra file. Cách này phù hợp để lưu nhanh 1 lần, nhưng không linh hoạt khi cần lặp qua nhiều trang hay xử lý lỗi — đó là lý do bước tiếp theo ta chuyển sang Python.

In [ ]:
!curl -s -H "X-API-Key: {API_KEY}" "https://xosoapi.online/api/v1/vietnam/draws" \
  | python -m json.tool > draws.json

## 3. Gọi API bằng Python (`requests`)

`curl` tốt để thăm dò, nhưng để xử lý logic (lặp trang, retry, lọc dữ liệu, lưu DataFrame...) ta cần code Python thật sự. Thư viện `requests` cho phép:

- Gửi header (`X-API-Key`) và query params (`page`, `limit`) một cách tường minh.
- Bắt lỗi HTTP (`response.raise_for_status()`) thay vì phải tự đọc status code bằng mắt.
- Parse JSON trực tiếp thành `dict`/`list` Python (`response.json()`).

In [ ]:
import requests

headers = {
    "X-API-Key": API_KEY,
    "Accept": "application/json",
}

query_params = {
    "limit": 10,
    "page": 1,
}

try:
    response = requests.get(API_URL, headers=headers, params=query_params)
    response.raise_for_status()  # ném lỗi nếu status code là 4xx/5xx
    data = response.json()
    print(data)

except requests.exceptions.HTTPError as http_err:
    # 429 Too Many Requests: API đang giới hạn số lượt gọi trong một khoảng thời gian
    # (rate limit) — đây là lỗi rất thường gặp khi gọi API liên tục, không phải bug.
    print(f"HTTP error occurred: {http_err}")
except Exception as err:
    print(f"Other error occurred: {err}")

## 4. Gọi nhiều trang (phân trang) kèm retry khi bị rate limit

API trả tối đa `limit` bản ghi mỗi lần gọi, nên cần lặp qua nhiều `page` để lấy hết dữ liệu mong muốn. Vì gọi liên tục dễ dính lỗi `429`, ta thêm cơ chế **retry với backoff**: khi bị rate limit, chờ một khoảng thời gian tăng dần (1s, 2s, 4s...) rồi thử lại, thay vì bỏ cuộc ngay.

**Lưu ý quan trọng về cấu trúc dữ liệu:** mỗi response trả về có dạng `{"success": true, "data": [...], "meta": {...}}`. Nếu ta lưu nguyên cả response của từng trang vào một list, ta sẽ có một list *lồng nhau* (list của các response), không phải list *phẳng* các bản ghi xổ số. Ở bước phân tích dữ liệu (mục 6), ta cần duyệt trực tiếp từng bản ghi — vì vậy ngay tại đây ta chỉ lấy phần `data` của mỗi trang và `extend` (nối) vào một list phẳng duy nhất.

In [ ]:
import time

PAGES = 5
LIMIT = 100
all_draws = []  # <-- list PHẲNG chứa từng bản ghi xổ số, không phải từng response theo trang

for page in range(1, PAGES + 1):
    query_params = {"limit": LIMIT, "page": page}

    for attempt in range(3):  # thử tối đa 3 lần cho mỗi trang nếu bị rate limit
        try:
            response = requests.get(API_URL, headers=headers, params=query_params)
            response.raise_for_status()
            payload = response.json()

            # payload["data"] mới là list các bản ghi thật sự -> nối vào all_draws
            all_draws.extend(payload["data"])
            break

        except requests.exceptions.HTTPError as http_err:
            if response.status_code == 429:
                wait_seconds = 2 ** attempt  # backoff: 1s, 2s, 4s...
                print(f"Bị rate limit ở trang {page}, chờ {wait_seconds}s rồi thử lại...")
                time.sleep(wait_seconds)
                continue
            print(f"HTTP error occurred: {http_err}")
            break
        except Exception as err:
            print(f"Other error occurred: {err}")
            break

print(f"Đã lấy {len(all_draws)} bản ghi")

## 5. Lưu dữ liệu đã gộp ra file JSON

Lưu `all_draws` (list phẳng) ra `full_draws.json`. Việc tách bước "lấy dữ liệu" và "phân tích dữ liệu" ra 2 giai đoạn (qua một file trung gian) là một thói quen tốt: nếu bước phân tích ở dưới bị lỗi, ta không cần gọi lại API (tốn quota) mà chỉ cần đọc lại file đã lưu.

In [ ]:
import json

with open("full_draws.json", "w", encoding="utf-8") as f:
    json.dump(all_draws, f, ensure_ascii=False, indent=4)

## 6. Đọc lại file JSON và dựng bảng dữ liệu (pandas)

API này trả về xổ số của **nhiều khu vực** (Miền Bắc, Miền Nam, Vietlott...) trộn chung trong cùng một danh sách. Ta cần **lọc** (`if draw["province"]["region"]["name"] == "Miền Bắc"`) để chỉ giữ lại phần mình cần, rồi trích các trường muốn hiển thị (`results[0]` = Giải Đặc Biệt, `results[1]` = Giải Nhất, `results[2]` = Giải Nhì) thành các cột của DataFrame.

In [ ]:
import json
import pandas as pd

# Mở và đọc nội dung file JSON đã lưu ở bước trước (không cần gọi lại API)
with open("full_draws.json", "r", encoding="utf-8") as f:
    all_draws = json.load(f)

table_rows = []

for draw in all_draws:
    # Mỗi draw thuộc một tỉnh/miền khác nhau -> chỉ giữ lại "Miền Bắc"
    if draw["province"]["region"]["name"] == "Miền Bắc":
        row = {
            "date": draw["date"],
            # results[0] = Giải Đặc Biệt, results[1] = Giải Nhất, results[2] = Giải Nhì
            "KetquaDB": ", ".join(draw["results"][0]["values"]),
            "KetquaG1": ", ".join(draw["results"][1]["values"]),
            "KetquaG2": ", ".join(draw["results"][2]["values"]),
        }
        table_rows.append(row)

df = pd.DataFrame(table_rows)
df.head()

## Tổng kết — những gì cần nhớ khi làm việc với API thực tế

1. **Đọc tài liệu trước khi code**: xác định endpoint, auth, params, response schema, pagination, rate limit.
2. **Thăm dò bằng `curl`** trước khi viết code — rẻ và nhanh để xác nhận mọi thứ hoạt động.
3. **Luôn xử lý lỗi HTTP**, đặc biệt `429 Too Many Requests` — thêm retry/backoff thay vì để chương trình dừng đột ngột.
4. **Tách bước lấy dữ liệu và bước phân tích** qua một file trung gian (JSON/CSV) để đỡ tốn quota API khi debug phần phân tích.
5. **Kiểm tra kỹ cấu trúc dữ liệu** (nested vs flat) trước khi lưu — đây là lỗi rất dễ mắc phải khi ghép nhiều response phân trang lại với nhau.